<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Erste Agenten mit LangChain
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M03-Erste-Agenten"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---

In diesem Modul wird ein **erster vollständiger KI-Agenten** mit LangChain erstellt.

Ein Agent verbindet drei Komponenten:

| Komponente | Rolle | Beispiel |
|---|---|---|
| **LLM** | Entscheidet, welche Tools gebraucht werden | `gpt-4o-mini` |
| **Tools** | Führen konkrete Aktionen aus | `celsius_nach_fahrenheit()` |
| **System-Prompt** | Definiert Rolle und Verhalten | "Du bist ein hilfreicher Assistent..." |

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

# 2 | create_agent() – Moderne Agent-API
---

`create_agent()` ist die standardisierte API für ReAct-Agenten in LangChain 1.0+.

**Die wichtigsten Parameter:**

| Parameter | Typ | Beschreibung |
|---|---|---|
| `model` | `str` oder LLM | Welches LLM steuert den Agenten? |
| `tools` | `list[Tool]` | Welche Werkzeuge stehen zur Verfügung? |
| `system_prompt` | `str` | Wie soll der Agent auftreten und antworten? |

**Was intern passiert (TAO-Zyklus):**
1. Agent sendet Nutzeranfrage + Tool-Schemas an das LLM
2. LLM entscheidet: direkte Antwort oder Tool-Call?
3. Bei Tool-Call: Tool ausführen, Ergebnis zurückliefern
4. Zyklus wiederholen bis Antwort vollständig

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  Prozessdiagramm</font> </br></p>

diagram = '''
flowchart LR
    LLM["LLM\ngpt-4o-mini"]
    TOOLS["Tools\ncelsius_nach_fahrenheit\nist_buerozeit"]
    SYS["System-Prompt\nRolle & Verhalten"]
    AGENT["create_agent()"]
    RUN["agent.invoke(...)"]

    LLM --> AGENT
    TOOLS --> AGENT
    SYS --> AGENT
    AGENT --> RUN
'''

mermaid(diagram, width=600)

In [ ]:
# 2.2 Agent zusammensetzen mit create_agent()
from langchain.agents import create_agent
from langchain_core.tools import tool

# Tools definieren
@tool
def celsius_nach_fahrenheit(temperatur: float) -> float:
    """Rechnet eine Temperatur von Celsius in Fahrenheit um. Gibt den Fahrenheit-Wert zurueck."""
    return round(temperatur * 9 / 5 + 32, 2)

@tool
def ist_buerozeit(stunde: int) -> str:
    """Prueft ob eine Stunde des Tages (0-23) in die typische Buerozeit faellt (9-17 Uhr).
    Gibt 'Buerozeit' oder 'Ausserhalb Buerozeit' zurueck."""
    return "Buerozeit" if 9 <= stunde <= 17 else "Ausserhalb Buerozeit"

system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m03_agent_system_prompt.md", mode="S")

# Agent zusammensetzen
agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[celsius_nach_fahrenheit, ist_buerozeit],
    system_prompt=system_prompt,
)

mprint("**Agent bereit** mit folgenden Tools:")
for t in [celsius_nach_fahrenheit, ist_buerozeit]:
    mprint(f"- `{t.name}`: {t.description}")


# 3 | Erster Agent mit 2 Tools
---



Jetzt setzen wir alles zusammen: LLM, Tools und System-Prompt ergeben einen funktionierenden Agenten.

Der Agent entscheidet selbst, ob er ein Tool aufruft oder direkt antwortet – je nachdem, was die Anfrage erfordert.

In [ ]:
ergebnis = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Sind 21 Grad Celsius angenehm? Und: Es ist gerade 15 Uhr – ist das Bürozeit?"
    }]
})

mprint("## Agent-Antwort")
mprint(ergebnis["messages"][-1].content)

In [ ]:
mprint("## Verwendete Tool-Calls")
tools_gefunden = False
for msg in ergebnis["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        tools_gefunden = True
        for tc in msg.tool_calls:
            mprint(f"- `{tc['name']}({tc['args']})`")
if not tools_gefunden:
    mprint("*Kein Tool-Call – Agent hat direkt geantwortet.*")

# 4 | Agent-Verhalten testen
---



Testen Sie den Agenten mit verschiedenen Anfragen und beobachten Sie:
- Wann ruft er ein Tool auf?
- Wann antwortet er direkt ohne Tool?
- Was passiert bei Fragen, die mehrere Tools erfordern?

Diese Beobachtungen helfen, den System-Prompt gezielt zu optimieren.

In [ ]:
test_fragen = [
    ("Tool erwartet",  "Was sind 5 Grad Celsius in Fahrenheit?"),
    ("Tool erwartet",  "Ist 22 Uhr noch Bürozeit?"),
    ("Kein Tool",      "Erkläre mir kurz, was ein KI-Agent ist."),
    ("Beide Tools",    "Es ist 10 Uhr. Brauche ich eine Jacke bei 3 Grad Celsius?"),
]

for erwartung, frage in test_fragen:
    result = agent.invoke({"messages": [{"role": "user", "content": frage}]})

    tools_genutzt = [
        tc["name"]
        for msg in result["messages"]
        if hasattr(msg, "tool_calls") and msg.tool_calls
        for tc in msg.tool_calls
    ]

    mprint(f"\n---\n**Frage [{erwartung}]:** {frage}")
    mprint(f"**Antwort:** {result['messages'][-1].content}")
    if tools_genutzt:
        mprint(f"**Tools genutzt:** `{'`, `'.join(tools_genutzt)}`")
    else:
        mprint("**Kein Tool genutzt** – direkte LLM-Antwort")

# 5 | LangSmith: Agent-Trace
---



LangSmith visualisiert den vollständigen TAO-Zyklus eines Agenten:
- **Thought:** Welche Tool-Calls hat das LLM geplant?
- **Action:** Welche Parameter wurden übergeben?
- **Observation:** Was hat das Tool zurückgeliefert?

**Im Trace sehen Sie** den genauen Entscheidungsweg des Agenten – unschätzbar für Debugging und Optimierung des System-Prompts.

Das Tracing ist bereits in der Setup-Cell konfiguriert (`LANGSMITH_TRACING`, `LANGSMITH_ENDPOINT`, Projekt `M03-Erste-Agenten`).

In [ ]:
# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M03_Kap5_AgentTrace",
    "tags":     ["M03", "agent", "langsmith"]
}

# 2. with_config() anwenden, dann invoke()
trace_result = agent.with_config(**run_cfg).invoke({
    "messages": [{
        "role": "user",
        "content": "Wie viel Fahrenheit sind 100 Grad Celsius, und ist 16 Uhr Bürozeit?"
    }]
})

mprint("## Trace-Ergebnis")
mprint('---')
mprint(trace_result["messages"][-1].content)

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M03-Erste-Agenten", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.




<p><font color='black' size="5">
Erster eigener Agent
</font></p>



Erstellen Sie einen Agenten für einen Prozess aus Ihrem Arbeitsumfeld – nutzen Sie Ihre Tools aus *Tool Use & Function Calling* oder definieren Sie neue.

**Mindest-Anforderungen:**
- Mindestens 2 selbst definierte Tools
- Einen passenden System-Prompt für die Rolle des Agenten
- Mindestens 3 Test-Anfragen (davon 1 ohne Tool-Nutzung)

**Optionale Teilaufgaben:**
- Fügen Sie ein drittes Tool hinzu und beobachten Sie, wann der Agent es wählt
- Passen Sie den System-Prompt an und beschreiben Sie, wie sich das Verhalten ändert
- Aktivieren Sie LangSmith-Tracing und beschreiben Sie, was Sie im TAO-Zyklus sehen

---
### 🔍 Selbstcheck mit `assert`

`assert` prüft eine Bedingung — ist sie `False`, stoppt Python mit einem `AssertionError` und zeigt die Fehlermeldung an:

```python
assert bedingung, "Fehlermeldung"

assert 2 + 2 == 4, "Rechnung falsch"    # ✅ kein Fehler
assert len("Hi") > 5, "Text zu kurz"   # ❌ AssertionError: Text zu kurz
```

**So nutzen Sie den Selbstcheck:**
1. Schreiben Sie Ihre Lösung in die Zellen über diesem Block
2. Speichern Sie Ihren Agenten in der Variable **`mein_agent`** (oder passen Sie den Namen in der Selbstcheck-Zelle an)
3. Führen Sie die Zelle unten aus — ein `✅` zeigt: Mindestanforderungen erfüllt


In [ ]:
#@title ✅ Selbstcheck – Erster eigener Agent  { display-mode: "form" }
# ► Speichere deinen Agenten in der Variable 'mein_agent' – oder passe die erste Zeile an.

_ag = mein_agent  # ← Variablennamen anpassen

assert hasattr(_ag, "invoke"), \
    "❌ Kein invoke() – wurde der Agent mit create_agent() erstellt?"

_r = _ag.invoke({"messages": [{"role": "user", "content": "Was kannst du für mich tun?"}]})
_antwort = _r["messages"][-1].content
assert len(_antwort) > 10, \
    f"❌ Antwort zu kurz ({len(_antwort)} Z.) – prüfe System-Prompt und Tools"

print(f"✅ Agent antwortet · {len(_antwort)} Zeichen")
print(f"   {_antwort[:120]}{'...' if len(_antwort) > 120 else ''}")
